In [ ]:
# Step 1: Install dependencies & set Hugging Face token
%pip install -q "datasets>=2.19.0" "huggingface_hub>=0.24"
import os
import getpass

# 直接设置Hugging Face token，跳过登录界面
hf_token = getpass.getpass("Paste your Hugging Face token: ")
os.environ['HF_TOKEN'] = hf_token
os.environ['HUGGINGFACE_HUB_TOKEN'] = hf_token

print("Hugging Face token set successfully!")

In [ ]:
# Step 2: Load FLARE-FPB test set and normalize labels
from datasets import load_dataset, Dataset

LABELS = ["negative", "neutral", "positive"]

ds_raw = load_dataset("TheFinAI/flare-fpb", split="test")
print("Loaded flare-fpb test:", len(ds_raw), "columns:", ds_raw.column_names)

_alias = {"pos": "positive", "neg": "negative", "neu": "neutral",
          "bullish": "positive", "bearish": "negative"}

def _norm_label(v):
    if v is None: 
        return None
    if isinstance(v, (int, float)) or (isinstance(v, str) and v.isdigit()):
        i = int(v)
        return LABELS[i] if 0 <= i < len(LABELS) else None
    s = str(v).strip().lower()
    s = _alias.get(s, s)
    return s if s in LABELS else None

def _map_row(x):
    text = x.get("text") or x.get("sentence") or x.get("content") or x.get("input") or ""
    lab = _norm_label(x.get("label", x.get("labels", x.get("answer"))))
    return {"text": text, "choices": LABELS, "answer": lab}

ds = Dataset.from_list([{**r, **_map_row(r)} for r in ds_raw])
bad = [i for i, r in enumerate(ds) if r["answer"] not in LABELS]
print("Samples with unusable label:", len(bad))
assert len(bad) == 0, "Found unparseable labels; please check the field mapping."

In [ ]:
# Step 3: Install dependencies, configure Gemini API, and record experiment metadata
%pip install -q "google-generativeai>=0.8.0" "httpx==0.27.2" "httpcore==1.0.5" \
               "pandas>=2.2.2" "tqdm>=4.66.4" "requests>=2.31.0"

import os, getpass, json, time, platform
from importlib.metadata import version, PackageNotFoundError

# Gemini 2.5 Flash配置
MODEL = "gemini-2.5-flash"
BASE_URL = "https://generativelanguage.googleapis.com/v1beta"  # Gemini API端点

api_key = os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY")
if not api_key:
    api_key = getpass.getpass("Paste your Gemini API key: ")
os.environ["GEMINI_API_KEY"] = api_key

# Gemini适配：调整文件命名
run_tag = f"flare_fpb_{MODEL.replace('.', '_').replace('-', '_')}"
save_dir = "/content"
pred_path = f"{save_dir}/{run_tag}_predictions.csv"
meta_path = f"{save_dir}/{run_tag}_metadata.json"

def ver(pkg: str) -> str:
    try:
        return version(pkg)
    except PackageNotFoundError:
        return "not-installed"

# Gemini适配：更新元数据
meta = {
    "dataset": "TheFinAI/flare-fpb",
    "split": "test",
    "labels": list(LABELS),
    "model": MODEL,
    "provider": "Google",
    "model_version": "flash",
    "google_genai_sdk": ver("google-generativeai"),
    "httpx": ver("httpx"),
    "httpcore": ver("httpcore"),
    "datasets_version": ver("datasets"),
    "pandas": ver("pandas"),
    "tqdm": ver("tqdm"),
    "time_utc": time.strftime("%Y-%m-%d %H:%M:%S", time.gmtime()),
    "python": platform.python_version(),
    "base_url": BASE_URL,
    "note": "Gemini 2.5 Flash model evaluation"
}

os.makedirs(save_dir, exist_ok=True)
with open(meta_path, "w") as f:
    json.dump(meta, f, indent=2)

print("Meta saved ->", meta_path)
print("MODEL:", MODEL, "| BASE_URL:", BASE_URL)
print("GEMINI_API_KEY is set:", bool(os.environ.get("GEMINI_API_KEY")))

In [ ]:
# Step 4: Inference & evaluation loop (Gemini 2.5 Flash适配)
import json, os, re, time, google.generativeai as genai
import pandas as pd
from tqdm import tqdm

# 配置Gemini API
genai.configure(api_key=os.environ["GEMINI_API_KEY"])

def _strip_code_fences(s: str) -> str:
    s = s.strip()
    if s.startswith("```"):
        s = re.sub(r"^```[a-zA-Z0-9_-]*\s*", "", s)
        s = re.sub(r"\s*```$", "", s)
    return s.strip()

def _make_user_text(sentence: str, choices=("",)):
    return (
        "Task: classify the sentence into exactly one of these labels: "
        f"{', '.join(choices)}.\n\n"
        f"Sentence: {sentence}\n\n"
        "Return ONLY a JSON object on a single line, exactly in this form:\n"
        "{\"label\":\"negative|neutral|positive\"}\n"
        "No code fences, no extra text, no explanation."
    )

def ask_gemini_once(sentence, choices=("negative", "neutral", "positive"), max_output_tokens=256):
    """使用Gemini API调用Gemini 2.5 Flash模型"""
    
    user_text = _make_user_text(sentence, choices)
    
    try:
        # 配置生成参数
        generation_config = {
            "temperature": 0.1,  # 低温度以获得更确定性的输出
            "top_p": 0.95,
            "top_k": 40,
            "max_output_tokens": max_output_tokens,
            "response_mime_type": "application/json",  # 强制JSON响应
        }
        
        # 创建模型实例
        model = genai.GenerativeModel(
            model_name=f"models/{MODEL}",
            generation_config=generation_config
        )
        
        # 生成响应
        response = model.generate_content(user_text)
        
        # 提取文本内容
        content = response.text
        content = _strip_code_fences(content)
        
        # 解析JSON响应
        obj = json.loads(content)
        lab = obj.get("label")
        
        if lab not in choices:
            raise RuntimeError(f"Invalid label {lab!r}; raw json: {obj}")
        
        return lab
        
    except json.JSONDecodeError as e:
        raise RuntimeError(f"JSON decode error: {str(e)} - Response: {content[:200]}")
    except Exception as e:
        raise RuntimeError(f"Gemini API error: {str(e)}")

def ask_gemini(sentence, choices=("negative", "neutral", "positive")):
    """带重试机制的Gemini调用"""
    for max_tokens in (256, 512, 1024):
        delay = 2.0
        for attempt in range(6):
            try:
                return ask_gemini_once(sentence, choices, max_output_tokens=max_tokens)
            except RuntimeError as e:
                msg = str(e).lower()
                
                # 处理限流和配额错误
                if "quota" in msg or "rate limit" in msg or "429" in msg:
                    time.sleep(delay)
                    delay = min(delay * 2, 90)
                    continue
                # 处理服务器错误
                if "server" in msg or "500" in msg or "503" in msg:
                    time.sleep(delay)
                    delay = min(delay * 2, 60)
                    continue
                # 处理内容安全问题
                if "safety" in msg or "blocked" in msg:
                    time.sleep(delay)
                    continue
                # JSON解析错误尝试重新调用
                if "json" in msg:
                    time.sleep(delay)
                    delay = min(delay * 1.5, 30)
                    continue
                raise
            except Exception as e:
                time.sleep(delay)
                delay = min(delay * 2, 60)
                if attempt < 5:
                    continue
                raise
    raise RuntimeError("Exhausted retries and token budgets for this sample.")

run_tag = f"flare_fpb_{MODEL.replace('.', '_').replace('-', '_')}"
save_dir = "/content"
pred_path = f"{save_dir}/{run_tag}_predictions.csv"
err_path = f"{save_dir}/{run_tag}_errors.csv"

rows_done = []
done_idx = set()
if os.path.exists(pred_path):
    old = pd.read_csv(pred_path)
    if "row_idx" in old.columns:
        rows_done = old.to_dict("records")
        done_idx = set(old["row_idx"].tolist())
        print(f"[resume] loaded {len(done_idx)} completed rows.")

err_rows = []
buf = []
save_every = 30

total = len(ds)
print(f"Starting Gemini 2.5 Flash model evaluation on {total} samples...")

for i in tqdm(range(total)):
    if i in done_idx:
        continue
    x = ds[i]
    text = x["text"]
    gold = x["answer"]

    try:
        pred = ask_gemini(text, LABELS)
        raw = json.dumps({"label": pred})
    except Exception as e:
        pred = "UNKNOWN"
        raw = f"ERROR: {type(e).__name__}: {e}"
        err_rows.append({"row_idx": i, "id": x.get("id", i), "error": raw, "text": text})

    buf.append({
        "row_idx": i,
        "id": x.get("id", i),
        "text": text,
        "pred_raw": raw,
        "pred": pred,
        "label": gold
    })

    if len(buf) % save_every == 0:
        out = pd.DataFrame(rows_done + buf).sort_values("row_idx")
        out.to_csv(pred_path, index=False)
        if err_rows:
            pd.DataFrame(err_rows).to_csv(err_path, index=False)
        print(f"[checkpoint] saved {len(out)}/{total} -> {pred_path}")

out = pd.DataFrame(rows_done + buf).sort_values("row_idx")
out.to_csv(pred_path, index=False)
if err_rows:
    pd.DataFrame(err_rows).to_csv(err_path, index=False)
print(f"[done] Gemini 2.5 Flash evaluation completed -> {pred_path}")
if os.path.exists(err_path):
    err_count = len(pd.read_csv(err_path)) if os.path.getsize(err_path) > 0 else 0
    print(f"[errors] {err_count} errors logged -> {err_path}")

In [ ]:
# Step 5: Install scikit-learn and compute evaluation metrics
%pip install -q scikit-learn

# Then compute Macro-F1 and Accuracy
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import json
import time

# 加载预测结果
df = pd.read_csv(pred_path).sort_values("row_idx").drop_duplicates("row_idx", keep="last")
ok = df[df["pred"] != "UNKNOWN"].copy()

print(f"Gemini 2.5 Flash Model Evaluation Results:")
print(f"Total samples: {len(df)}")
print(f"Successful predictions: {len(ok)}")
print(f"Failed predictions: {len(df) - len(ok)}")

if len(ok) > 0:
    # 计算评估指标
    f1_macro = f1_score(ok["label"], ok["pred"], labels=LABELS, average="macro", zero_division=0)
    f1_micro = f1_score(ok["label"], ok["pred"], labels=LABELS, average="micro", zero_division=0)
    f1_weighted = f1_score(ok["label"], ok["pred"], labels=LABELS, average="weighted", zero_division=0)
    accuracy = accuracy_score(ok["label"], ok["pred"])
    
    print("\n" + "="*50)
    print("EVALUATION RESULTS - Gemini 2.5 Flash")
    print("="*50)
    print(f"Accuracy:  {accuracy:.4f}")
    print(f"F1-Macro:  {f1_macro:.4f}")
    print(f"F1-Micro:  {f1_micro:.4f}")
    print(f"F1-Weighted: {f1_weighted:.4f}")
    
    # 详细分类报告
    print("\nDetailed Classification Report:")
    print(classification_report(ok["label"], ok["pred"], labels=LABELS, zero_division=0))
    
    # 混淆矩阵
    print("Confusion Matrix:")
    cm = confusion_matrix(ok["label"], ok["pred"], labels=LABELS)
    cm_df = pd.DataFrame(cm, index=LABELS, columns=LABELS)
    print(cm_df)
    
    # 保存评估结果
    eval_results = {
        "model": MODEL,
        "dataset": "TheFinAI/flare-fpb",
        "split": "test",
        "total_samples": len(df),
        "successful_predictions": len(ok),
        "failed_predictions": len(df) - len(ok),
        "accuracy": float(accuracy),
        "f1_macro": float(f1_macro),
        "f1_micro": float(f1_micro),
        "f1_weighted": float(f1_weighted),
        "evaluation_time": time.strftime("%Y-%m-%d %H:%M:%S", time.gmtime()),
        "confusion_matrix": cm.tolist(),
        "labels": LABELS
    }
    
    eval_path = f"{save_dir}/{run_tag}_evaluation_results.json"
    with open(eval_path, "w") as f:
        json.dump(eval_results, f, indent=2)
    print(f"\nEvaluation results saved -> {eval_path}")
    
else:
    print("No successful predictions to evaluate!")